# ITS AI Codellama-7B-Instruct QLoRA finetuning (RESEARCH INFERENCE)

**This notebook is the trimmed version of the original CODELLAMA-7B-INSTRUCT QLoRA.<b><b> This notebook is for only running the inference from the latest dataset for Pak Agus's research purpose.**

NOTE: Before we begin, ensure you already have access to T4 GPU in Kaggle. You can do so by verification using phone number or persona verification. If not, you won't be able to train the codellama using a GPU.

In [1]:
!nvidia-smi

!pip install -q --no-cache-dir \
  transformers==4.46.3 \
  accelerate==1.1.1 \
  peft==0.14.0 \
  trl==0.12.2 \
  bitsandbytes==0.48.1 \
  datasets==3.1.0 \
  huggingface_hub==0.26.2
 
!pip uninstall -y triton

Thu Aug 20 07:01:45 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   46C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
import os
for root, dirs, files in os.walk("/kaggle/input"):
    for f in files:
        print(os.path.join(root, f))

/kaggle/input/datasets/hanzfr/soal-ujian/soal_ujian.json
/kaggle/input/datasets/hanzfr/codellama-qlora-final-model-v2-fix/final_model/adapter_model.safetensors
/kaggle/input/datasets/hanzfr/codellama-qlora-final-model-v2-fix/final_model/train_data.jsonl
/kaggle/input/datasets/hanzfr/codellama-qlora-final-model-v2-fix/final_model/training_args.bin
/kaggle/input/datasets/hanzfr/codellama-qlora-final-model-v2-fix/final_model/adapter_config.json
/kaggle/input/datasets/hanzfr/codellama-qlora-final-model-v2-fix/final_model/README.md
/kaggle/input/datasets/hanzfr/codellama-qlora-final-model-v2-fix/final_model/tokenizer.json
/kaggle/input/datasets/hanzfr/codellama-qlora-final-model-v2-fix/final_model/val_split.jsonl
/kaggle/input/datasets/hanzfr/codellama-qlora-final-model-v2-fix/final_model/tokenizer_config.json
/kaggle/input/datasets/hanzfr/codellama-qlora-final-model-v2-fix/final_model/__huggingface_repos__.json
/kaggle/input/datasets/hanzfr/codellama-qlora-final-model-v2-fix/final_model/tr

In [3]:
import json

# change the path according to your preferred set filepath
with open("/kaggle/input/datasets/hanzfr/soal-ujian/soal_ujian.json") as f:
    soal = json.load(f)
with open("/kaggle/input/datasets/hanzfr/nilai-ujian/nilai_ujian.json") as f:
    nilai = json.load(f)

soal_by_id = {s["id"]: s for s in soal}

def normalize_scores(nilai_dict):
    if max(nilai_dict.values()) <= 10:
        return {k: v * 10 for k, v in nilai_dict.items()}
    return nilai_dict

dataset = []
for n in nilai:
    q = soal_by_id.get(n["id_soal"])
    if q is None:
        continue
    scores = normalize_scores(n["nilai"])
    avg = round(sum(scores.values()) / len(scores), 2)
    dataset.append({
        "id_soal": n["id_soal"],
        "soal": q["soal"],
        "expected_output": q["expected_output"],
        "kode_siswa": n["kode_siswa"],
        "level_siswa": n["level_siswa"],
        "nilai": scores,
        "nilai_avg": avg,
        "feedback": n["feedback"],
    })

print(f"Usable examples: {len(dataset)}")

def format_example(ex):
    prompt = (
        f"Soal: {ex['soal']}\n"
        f"Output yang diharapkan: {ex['expected_output']}\n\n"
        f"Kode siswa:\n```python\n{ex['kode_siswa']}\n```\n\n"
        f"Nilai kode siswa ini dan berikan feedback."
    )
    response = (
        f"Penilaian:\n"
        + "\n".join(f"- {k}: {v}" for k, v in ex["nilai"].items())
        + f"\n\nRata-rata: {ex['nilai_avg']}\n\nFeedback: {ex['feedback']}"
    )
    return {"id_soal": ex["id_soal"], "text": f"<s>[INST] {prompt} [/INST] {response} </s>"}

formatted = [format_example(ex) for ex in dataset]

with open("/kaggle/working/train_data.jsonl", "w") as f:
    for row in formatted:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

Usable examples: 1512


In [4]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("codellama/CodeLlama-7b-Instruct-hf")

lengths = []
with open("/kaggle/working/train_data.jsonl") as f:
    for line in f:
        row = json.loads(line)
        lengths.append(len(tokenizer(row["text"])["input_ids"]))

import statistics
print("min/max:", min(lengths), max(lengths))
print("mean:", statistics.mean(lengths))
print("p95:", sorted(lengths)[int(len(lengths)*0.95)])

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/411 [00:00<?, ?B/s]

min/max: 274 1151
mean: 510.9298941798942
p95: 842


In [5]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

# Load base model WITHOUT 4-bit quantization this time — merging requires full precision
base_model = AutoModelForCausalLM.from_pretrained(
    "codellama/CodeLlama-7b-Instruct-hf",
    torch_dtype=torch.float16,
    device_map={"": 0},
)

MODEL_PATH = "/kaggle/input/datasets/hanzfr/codellama-qlora-final-model-v2-fix/final_model"
model = PeftModel.from_pretrained(base_model, MODEL_PATH)

merged_model = model.merge_and_unload()
merged_model.save_pretrained("/kaggle/working/merged_model", safe_serialization=True)

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
tokenizer.save_pretrained("/kaggle/working/merged_model")

config.json:   0%|          | 0.00/646 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/9.98G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/3.50G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

('/kaggle/working/merged_model/tokenizer_config.json',
 '/kaggle/working/merged_model/special_tokens_map.json',
 '/kaggle/working/merged_model/tokenizer.model',
 '/kaggle/working/merged_model/added_tokens.json',
 '/kaggle/working/merged_model/tokenizer.json')

In [8]:
import json, re

eval_rows = [json.loads(l) for l in open("/kaggle/input/datasets/hanzfr/research-dataset-2/inference_eval_data.jsonl")]

N_ITERATIONS = 10
comparison_results = []

for idx, row in enumerate(eval_rows[0:269]):
    print(f"Processing example {idx+1}/265 (id_soal={row['id_soal']})...")
    prompt = row["text"].split("[/INST]")[0] + "[/INST]"
    ground_truth = row["text"].split("[/INST]")[1].replace("</s>", "").strip()

    generations = []
    for _ in range(N_ITERATIONS):
        inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
        output = model.generate(
            **inputs, max_new_tokens=250, temperature=0.7, do_sample=True,
            eos_token_id=tokenizer.eos_token_id, pad_token_id=tokenizer.eos_token_id,
        )
        generated_full = tokenizer.decode(output[0], skip_special_tokens=True)
        generated = generated_full.split("[/INST]")[1].strip() if "[/INST]" in generated_full else generated_full
        generations.append(generated)

    comparison_results.append({
        "id_soal": row["id_soal"],
        "topic": row["topic"],
        "level_siswa": row["level_siswa"],
        "ground_truth": ground_truth,
        "generations": generations, 
    })

with open("/kaggle/working/comparison_results.json", "w", encoding="utf-8") as f:
    json.dump(comparison_results, f, ensure_ascii=False, indent=2)

Processing example 1/20 (id_soal=24)...


In [9]:
import json

with open("/kaggle/working/comparison_results.json", encoding="utf-8") as f:
    results = json.load(f)

print(f"Total examples: {len(results)}")
print(json.dumps(results[0], ensure_ascii=False, indent=2))  # preview first example

Total examples: 1
{
  "id_soal": 24,
  "topic": "Sintaks Dasar Pemrograman - Konversi Bilangan (Basis Angka)",
  "level_siswa": "Advance",
  "ground_truth": "Penilaian:\n- fungsionalitas: 95\n- logika: 92\n- syntax: 95\n- code_style: 88\n- dokumentasi: 80\n- konsep: 88\n\nRata-rata: 89.67\n\nFeedback: Kode sudah solid: ada validasi rentang input dengan raise ValueError yang jelas, docstring lengkap menjelaskan perilaku dan exception, tabel konversi tersusun rapi dan mudah dibaca, serta memakai guard __main__ yang merupakan praktik baik dalam pengembangan skrip Python. Hasil untuk 14 benar. Sedikit catatan: bisa ditambah unit test terpisah untuk memverifikasi berbagai kasus batas (1, 3999, 4, 9, 40, dst).",
  "generations": [
    "Penilaian:\n- fungsionalitas: 95\n- logika: 90\n- syntax: 95\n- code_style: 85\n- dokumentasi: 80\n- konsep: 85\n\nRata-rata: 89.17\n\nFeedback: Solusi sangat baik: dokumentasi jelas, validasi input memakai guard (if __name__ == \"__main__\"). Struktur tabel d